# Week 4: model comparison with Week 3

Two views:

1. Fair pixel head-to-head on the same Week 4 balanced eval set (`n_RES = n_NRES = 29,798`).
2. Headline ladder: Week 3 reported LORO F1 (imbalanced / 3-class protocols) vs Week 4 balanced train.

Artifacts: `outputs/week4_model_comparison/`


In [1]:
from pathlib import Path
import json
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = HERE
for p in [HERE, *HERE.parents]:
    if (p / "outputs" / "week4_model_comparison" / "summary.json").exists():
        ROOT = p
        break

CMP = ROOT / "outputs" / "week4_model_comparison"
board = pd.read_csv(CMP / "comparison_balanced_eval.csv")
ladder = pd.read_csv(CMP / "f1_ladder_cross_week.csv")
summary = json.loads((CMP / "summary.json").read_text())

print(f"Balanced eval n={summary['balanced_eval_n']:,}  "
      f"RES={summary['balanced_eval_res']:,}  NRES={summary['balanced_eval_nres']:,}")
display(
    board[["model", "protocol_train", "macro_f1_week3_reported",
           "macro_f1_balanced_eval", "mae_mean_balanced"]]
    .sort_values("macro_f1_balanced_eval", ascending=False)
    .round(4)
)


Balanced eval n=59,596  RES=29,798  NRES=29,798


,model,protocol_train,macro_f1_week3_reported,macro_f1_balanced_eval,mae_mean_balanced
0,balanced_train (W4),"W4 balanced, no SMOTE",0.7041,0.7043,0.2807
1,fraction_regression (W3),W3 imbalanced+SMOTE,0.5715,0.6999,0.4020
2,pixel_ae (W3),W3 imbalanced+SMOTE,0.5633,0.6972,0.4013
3,pixel_3class_reg (W3),W3 + empty class,0.5030,0.6675,0.2776
4,neighborhood_k3_reg (W3),W3 3x3 CNN + empty,0.3454,0.5006,0.3027


## 1. Fair comparison on balanced pixels

All models scored on the same Week 4 balanced `pixel_id` set. GHSL = nearest GHS-BUILT-C FUN value at the AE cell center.

| Model | Train protocol | macro-F1 (balanced eval) | mean frac MAE |
|---|---|---:|---:|
| balanced_train (W4) | 1:1 RES/NRES, no SMOTE | 0.704 | 0.281 |
| fraction_regression (W3) | imbalanced + SMOTE | 0.700 | 0.402 |
| pixel_ae LR (W3) | imbalanced + SMOTE | 0.697 | 0.401 |
| GHSL (pixel) | FUN @ AE centers | 0.692 | — |
| pixel_3class soft (W3) | + empty class | 0.668 | 0.278 |
| neighborhood 3×3 (W3) | CNN + empty | 0.501 | 0.303 |

Binary F1: AE heads and GHSL are within ~0.01 on this set (0.704 vs 0.692).
Fractions: Week 3 soft 3-class still has the best MAE (0.278).

![F1 on balanced eval](../../outputs/week4_model_comparison/figures/f1_on_balanced_eval.png)

![GHSL vs AE](../../outputs/week4_ghsl_pixel/figures/ghsl_vs_ae_balanced_ladder.png)

![MAE on balanced eval](../../outputs/week4_model_comparison/figures/mae_on_balanced_eval.png)


## 2. Why Week 3 headlines looked lower

Week 3 reported F1 on imbalanced holdouts (~8.5:1 NRES:RES). The same OOF preds, restricted to the balanced subset, rise ~0.13–0.14 F1 for AE/Ridge.

| Model | Week 3 reported F1 | F1 on W4 balanced subset | Δ |
|---|---:|---:|---:|
| pixel_ae | 0.563 | 0.697 | +0.13 |
| fraction_regression | 0.571 | 0.700 | +0.13 |
| pixel_3class soft | 0.503 (3-class) | 0.668 (binary decode) | n/a |
| neighborhood 3×3 | 0.345 | 0.501 | +0.16 |
| balanced_train (W4) | n/a | 0.704 | trained + eval on 1:1 |

![Protocol shift](../../outputs/week4_model_comparison/figures/f1_protocol_shift.png)


## 3. Cross-week headline ladder (different units)

Building-scale scores are not the same task as 10 m pixels. Shown only for context.

| Model | Unit | Reported macro-F1 |
|---|---|---:|
| Polygon AE CNN | building | 0.746 |
| balanced_train LR (W4) | pixel, 1:1 | 0.704 |
| GHSL (pixel, W4 balanced) | pixel, 1:1 | 0.692 |
| GHSL | building | 0.672 |
| fraction_regression (W3) | pixel, imbalanced | 0.571 |
| pixel_ae LR (W3) | pixel, imbalanced | 0.563 |
| GHSL (W3 pixel protocol) | pixel, imbalanced | 0.512 |
| pixel_3class soft (W3) | pixel + empty | 0.503 |
| neighborhood 3×3 (W3) | pixel + empty | 0.345 |

![Cross-week ladder](../../outputs/week4_model_comparison/figures/cross_week_f1_ladder.png)

![GHSL by metro](../../outputs/week4_ghsl_pixel/figures/ghsl_f1_by_metro_balanced.png)


## Takeaways

1. Balancing the eval set (and train) is the main Week 4 win for binary F1. Week 4 LR hits 0.704 without SMOTE; GHSL hits 0.692 on the same cells.
2. On that balanced set, Week 3 AE/Ridge OOF already reach ~0.70; the old 0.56 headlines came from imbalance, not weak embeddings. GHSL’s Week 3 pixel F1 (~0.51) jumps the same way.
3. For fraction MAE, Week 3 soft 3-class (0.278) still slightly beats Week 4 Ridge (0.281).
4. Building Polygon AE (0.75) remains the highest absolute F1, but it is a different unit.
